In [1]:
import os 

from pathlib import Path
from shapely.geometry import Polygon, mapping

import json
import pandas as pd
import geopandas as gpd
import numpy as np
from shapely.geometry import mapping

from pystac_client import Client
import planetary_computer as pc
import stackstac
import rioxarray
import rasterio

In [2]:
# 1. Configuración general

BASE_DIR = Path(os.path.abspath(os.path.join(os.getcwd(), os.pardir)))
RASTER_DIR = BASE_DIR / "raster"
LABELS_DIR = BASE_DIR / "labels"
METADATA_DIR = BASE_DIR / "metadata"

for folder in [RASTER_DIR, LABELS_DIR, METADATA_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

In [3]:
# 2. AOI preliminar - Salar de Olaroz
# AOI amplio para incluir salar, bordes, zonas húmedas,
# infraestructura, piletas y suelos periféricos.

olaroz_polygon = Polygon([
    (-66.86, -23.68),
    (-66.50, -23.68),
    (-66.50, -23.25),
    (-66.86, -23.25),
    (-66.86, -23.68),
])

In [4]:
aoi_gdf = gpd.GeoDataFrame(
    {
        "name": ["Salar de Olaroz - AOI preliminar"],
        "purpose": ["Mentoría - clasificación de cobertura terrestre"],
    },
    geometry=[olaroz_polygon],
    crs="EPSG:4326",
)

aoi_path = LABELS_DIR / "olaroz_aoi.geojson"
aoi_gdf.to_file(aoi_path, driver="GeoJSON")

print(f"AOI guardado en: {aoi_path}")

AOI guardado en: C:\Users\pablonicolasr\Desktop\pablonicolas\educacion_formal\mentodatos2026\repo\mentodatos_pdis\labels\olaroz_aoi.geojson


In [5]:
# 3. Parámetros auditables de búsqueda

COLLECTION = "sentinel-2-l2a"
DATE_RANGE = "2025-06-01/2025-09-30"   # época seca
MAX_CLOUD = 20                         # nubosidad máxima
BANDS = ["B01", "B02", "B03", "B04", "B05", "B06", "B07", "B08", "B8A", "B09", "B11", "B12", "SCL"]

# Bandas Sentinel-2 L2A:
# B01  = Coastal Aerosol / Aerosol costero          | 60 m
# B02  = Blue / Azul                                | 10 m
# B03  = Green / Verde                              | 10 m
# B04  = Red / Rojo                                 | 10 m
# B05  = Vegetation Red Edge 1 / Borde rojo 1       | 20 m
# B06  = Vegetation Red Edge 2 / Borde rojo 2       | 20 m
# B07  = Vegetation Red Edge 3 / Borde rojo 3       | 20 m
# B08  = NIR / Infrarrojo cercano                   | 10 m
# B08A = Narrow NIR / NIR estrecho                  | 20 m
# B09  = Water Vapour / Vapor de agua               | 60 m
# B11  = SWIR 1 / Infrarrojo onda corta 1           | 20 m
# B12  = SWIR 2 / Infrarrojo onda corta 2           | 20 m
# SCL  = Scene Classification Layer                 | 20 m


In [6]:
# 4. Consulta STAC

catalog = Client.open("https://planetarycomputer.microsoft.com/api/stac/v1")

search = catalog.search(
    collections=[COLLECTION],
    intersects=mapping(olaroz_polygon),
    datetime=DATE_RANGE,
    query={"eo:cloud_cover": {"lt": MAX_CLOUD}},
    max_items=100,
)

items = list(search.items())

if not items:
    raise RuntimeError("No se encontraron imágenes con esos filtros. Probá ampliar fechas o nubosidad.")

print(f"Imágenes candidatas encontradas: {len(items)}")

Imágenes candidatas encontradas: 56


In [7]:
# 5. Guardar metadatos de todas las escenas candidatas

records = []

for item in items:
    props = item.properties

    records.append({
        "item_id": item.id,
        "datetime": props.get("datetime"),
        "platform": props.get("platform"),
        "constellation": props.get("constellation"),
        "mgrs_tile": props.get("s2:mgrs_tile"),
        "eo_cloud_cover": props.get("eo:cloud_cover"),
        "processing_baseline": props.get("s2:processing_baseline"),
        "collection": COLLECTION,
        "date_range_filter": DATE_RANGE,
        "cloud_filter": MAX_CLOUD,
    })

metadata_df = pd.DataFrame(records)
metadata_df = metadata_df.sort_values("eo_cloud_cover", ascending=True)

candidates_csv = METADATA_DIR / "sentinel2_olaroz_candidate_scenes.csv"
metadata_df.to_csv(candidates_csv, index=False, encoding="utf-8")

print(f"Metadatos de escenas candidatas guardados en: {candidates_csv}")

Metadatos de escenas candidatas guardados en: C:\Users\pablonicolasr\Desktop\pablonicolas\educacion_formal\mentodatos2026\repo\mentodatos_pdis\metadata\sentinel2_olaroz_candidate_scenes.csv


In [8]:
# 6. Seleccionar la escena con menor nubosidad

selected_id = metadata_df.iloc[0]["item_id"]

selected_item = None
for item in items:
    if item.id == selected_id:
        selected_item = item
        break

if selected_item is None:
    raise RuntimeError("No se pudo recuperar la escena seleccionada.")

selected_item = pc.sign(selected_item)

selected_json = METADATA_DIR / "selected_sentinel2_olaroz_item.json"
with open(selected_json, "w", encoding="utf-8") as f:
    json.dump(selected_item.to_dict(), f, ensure_ascii=False, indent=2)

selected_summary = metadata_df.iloc[[0]].copy()
selected_csv = METADATA_DIR / "selected_sentinel2_olaroz_scene.csv"
selected_summary.to_csv(selected_csv, index=False, encoding="utf-8")

print("Escena seleccionada:")
print(selected_summary.T)
print(f"STAC JSON guardado en: {selected_json}")

Escena seleccionada:
                                                                    12
item_id              S2B_MSIL2A_20250827T142719_R053_T19KGQ_2025082...
datetime                                   2025-08-27T14:27:19.024000Z
platform                                                   Sentinel-2B
constellation                                               Sentinel 2
mgrs_tile                                                        19KGQ
eo_cloud_cover                                                0.034071
processing_baseline                                              05.11
collection                                              sentinel-2-l2a
date_range_filter                                2025-06-01/2025-09-30
cloud_filter                                                        20
STAC JSON guardado en: C:\Users\pablonicolasr\Desktop\pablonicolas\educacion_formal\mentodatos2026\repo\mentodatos_pdis\metadata\selected_sentinel2_olaroz_item.json


In [9]:
# 7. Descargar/leer bandas y crear stack multibanda

bounds = aoi_gdf.total_bounds  # minx, miny, maxx, maxy en EPSG:4326

stack = stackstac.stack(
    [selected_item],
    assets=BANDS,
    bounds_latlon=bounds,
    resolution=20,
    dtype="uint16",
    fill_value=np.uint16(0),
    rescale=False,
)

# Quitar dimensión temporal
img = stack.squeeze(dim="time", drop=True)

# Convertir AOI a lista de geometrías, no FeatureCollection
geometries = [
    mapping(geom)
    for geom in aoi_gdf.geometry
    if geom is not None and not geom.is_empty
]

# Recortar exactamente por AOI
img_clip = img.rio.clip(
    geometries,
    crs=aoi_gdf.crs,
    drop=True,
)

# Forzar carga en memoria antes de escribir
img_clip = img_clip.compute()

C:\Python39\lib\site-packages\dask\array\chunk.py:278: RuntimeWarning: invalid value encountered in cast
  return x.astype(astype_dtype, **kwargs)


In [10]:
# 8. Exportar GeoTIFF multibanda

out_tif = RASTER_DIR / "olaroz_sentinel2_l2a_seca_2024_multiband.tif"
img_clip.rio.to_raster(out_tif)

# Agregar descripciones y tags al GeoTIFF
with rasterio.open(out_tif, "r+") as dst:
    for idx, band_name in enumerate(BANDS, start=1):
        dst.set_band_description(idx, band_name)

    dst.update_tags(
        area="Salar de Olaroz, Jujuy, Argentina",
        collection=COLLECTION,
        date_range_filter=DATE_RANGE,
        cloud_filter=str(MAX_CLOUD),
        selected_item_id=selected_item.id,
        purpose="Mentoria - clasificación clásica de cobertura terrestre",
        bands=",".join(BANDS),
        resolution_meters="20",
    )

print(f"GeoTIFF multibanda guardado en: {out_tif}")

GeoTIFF multibanda guardado en: C:\Users\pablonicolasr\Desktop\pablonicolas\educacion_formal\mentodatos2026\repo\mentodatos_pdis\raster\olaroz_sentinel2_l2a_seca_2024_multiband.tif


In [11]:
print("BANDS:", BANDS)
print("Cantidad en BANDS:", len(BANDS))

print("Dimensiones img_clip:", img_clip.dims)
print("Tamaños img_clip:", img_clip.sizes)

with rasterio.open(out_tif) as src:
    print("Cantidad real de bandas del TIFF:", src.count)

BANDS: ['B01', 'B02', 'B03', 'B04', 'B05', 'B06', 'B07', 'B08', 'B8A', 'B09', 'B11', 'B12', 'SCL']
Cantidad en BANDS: 13
Dimensiones img_clip: ('band', 'y', 'x')
Tamaños img_clip: Frozen({'band': 13, 'y': 2411, 'x': 1878})
Cantidad real de bandas del TIFF: 13


In [12]:
# 9. Crear protocolo de adquisición en Markdown

protocol = f"""# Protocolo de adquisición - Salar de Olaroz

## Área de estudio
Salar de Olaroz, departamento de Susques, provincia de Jujuy, Argentina.

## Fuente de datos
Microsoft Planetary Computer STAC API.

## Colección
`{COLLECTION}`

## Producto
Sentinel-2 Level-2A Surface Reflectance.

## Criterios de búsqueda
- AOI: `labels/olaroz_aoi.geojson`
- Rango temporal: `{DATE_RANGE}`
- Nubosidad máxima: `{MAX_CLOUD}%`
- Bandas exportadas: `{",".join(BANDS)}`
- Resolución de trabajo: 20 metros
- Criterio de selección: escena candidata con menor `eo:cloud_cover`

## Archivos generados
- Raster multibanda: `raster/{out_tif.name}`
- Escenas candidatas: `metadata/{candidates_csv.name}`
- Escena seleccionada CSV: `metadata/{selected_csv.name}`
- Escena seleccionada STAC JSON: `metadata/{selected_json.name}`
- AOI: `labels/{aoi_path.name}`

## Uso previsto
Este dataset se utilizará para una práctica de análisis de datos y aprendizaje automático clásico.
Los alumnos podrán extraer valores espectrales e índices desde el raster usando polígonos GeoJSON etiquetados.
"""

protocol_path = METADATA_DIR / "acquisition_protocol.md"
protocol_path.write_text(protocol, encoding="utf-8")

print(f"Protocolo guardado en: {protocol_path}")
print("Proceso finalizado correctamente.")

Protocolo guardado en: C:\Users\pablonicolasr\Desktop\pablonicolas\educacion_formal\mentodatos2026\repo\mentodatos_pdis\metadata\acquisition_protocol.md
Proceso finalizado correctamente.
